Giai đoạn 5: Evaluation

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv('../data/processed/movies_features.csv')

with open('../models_artifacts/user_item_matrix.pkl', 'rb') as f:
    user_item_matrix = pickle.load(f)
with open('../models_artifacts/tfidf_matrix.pkl', 'rb') as f:
    tfidf_matrix = pickle.load(f)
with open('../models_artifacts/content_based_candidate_indices.pkl', 'rb') as f:
    candidate_indices = pickle.load(f)

print("user_item_matrix:", user_item_matrix.shape)

user_item_matrix: (671, 3493)


Bước 5.1. Tách train/test theo từng user

In [4]:
np.random.seed(42)

ratings_long = user_item_matrix.stack().reset_index()
ratings_long.columns = ['userId', 'tmdbId', 'rating']
ratings_long = ratings_long[ratings_long['rating'] > 0].reset_index(drop=True)

print("Tổng số rating thực tế:", ratings_long.shape[0])

# Xáo trộn toàn bộ dữ liệu trước (để việc chọn 20% đầu mỗi user là ngẫu nhiên)
ratings_long = ratings_long.sample(frac=1, random_state=42).reset_index(drop=True)

# Đánh số thứ tự rating trong từng user (sau khi đã xáo trộn) + tổng số rating của user đó
ratings_long['user_rank'] = ratings_long.groupby('userId').cumcount()
ratings_long['user_count'] = ratings_long.groupby('userId')['userId'].transform('count')

# 20% đầu tiên (theo thứ tự đã xáo trộn) của mỗi user -> test, tối thiểu 1 rating/user
ratings_long['n_test'] = (ratings_long['user_count'] * 0.2).astype(int).clip(lower=1)
ratings_long['is_test'] = ratings_long['user_rank'] < ratings_long['n_test']

train_df = ratings_long[~ratings_long['is_test']].drop(columns=['user_rank', 'user_count', 'n_test', 'is_test'])
test_df = ratings_long[ratings_long['is_test']].drop(columns=['user_rank', 'user_count', 'n_test', 'is_test'])

print("Train:", train_df.shape[0], "| Test:", test_df.shape[0])
print("Số user có mặt trong test:", test_df['userId'].nunique())

Tổng số rating thực tế: 90015
Train: 72279 | Test: 17736
Số user có mặt trong test: 671


Bước 5.2. Dựng lại ma trận User-Item chỉ từ train, fit lại KNN

In [5]:
train_matrix = train_df.pivot_table(index='userId', columns='tmdbId', values='rating', fill_value=0)

# Đảm bảo train_matrix có đủ cột (tmdbId) giống user_item_matrix gốc, để so sánh nhất quán
train_matrix = train_matrix.reindex(columns=user_item_matrix.columns, fill_value=0)
train_matrix = train_matrix.reindex(index=user_item_matrix.index, fill_value=0)

print("train_matrix:", train_matrix.shape)

user_knn_train = NearestNeighbors(metric='cosine', algorithm='brute')
user_knn_train.fit(train_matrix.values)
print("Đã fit KNN trên tập train")

train_matrix: (671, 3493)
Đã fit KNN trên tập train


Bước 5.3: Hàm dự đoán rating (cho RMSE/MAE)

Khác với hàm xếp hạng top-N ở Giai đoạn 4 (chỉ cần thứ tự đúng), ở đây cần dự đoán ra 1 con số rating cụ thể để so sánh với rating thật trong tập test.

In [6]:
def predict_rating(user_id, tmdb_id, k_neighbors=10):
    if user_id not in train_matrix.index or tmdb_id not in train_matrix.columns:
        return None
    
    user_pos = train_matrix.index.get_loc(user_id)
    user_vector = train_matrix.iloc[user_pos].values.reshape(1, -1)
    
    distances, idx_knn = user_knn_train.kneighbors(user_vector, n_neighbors=k_neighbors + 1)
    similar_user_positions = idx_knn.flatten()[1:]
    similarities = 1 - distances.flatten()[1:]
    
    # Chỉ lấy rating của các neighbor ĐÃ rate phim này (bỏ qua neighbor chưa rate, khác với hàm ở Giai đoạn 4)
    neighbor_ratings = train_matrix.iloc[similar_user_positions][tmdb_id].values
    rated_mask = neighbor_ratings > 0
    
    if rated_mask.sum() == 0:
        return None  # không neighbor nào rate phim này -> không dự đoán được
    
    weights = similarities[rated_mask]
    ratings = neighbor_ratings[rated_mask]
    
    if weights.sum() < 1e-9:
        return ratings.mean()
    
    return np.dot(weights, ratings) / weights.sum()

In [7]:
# Test thử hàm trên 1 cặp (user, phim) có thật trong test set
sample = test_df.iloc[0]
pred = predict_rating(sample['userId'], sample['tmdbId'])
print(f"User {sample['userId']}, Phim {sample['tmdbId']}: thực tế={sample['rating']}, dự đoán={pred}")

User 184.0, Phim 32636.0: thực tế=2.0, dự đoán=None


Bước 5.4: Tính RMSE/MAE trên toàn bộ test set

In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

predictions, actuals = [], []
skipped = 0

for _, row in test_df.iterrows():
    pred = predict_rating(row['userId'], row['tmdbId'])
    if pred is not None:
        predictions.append(pred)
        actuals.append(row['rating'])
    else:
        skipped += 1

print(f"Dự đoán được: {len(predictions)} / {len(test_df)} (bỏ qua {skipped} do không đủ dữ liệu neighbor)")

rmse = np.sqrt(mean_squared_error(actuals, predictions))
mae = mean_absolute_error(actuals, predictions)
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Dự đoán được: 15269 / 17736 (bỏ qua 2467 do không đủ dữ liệu neighbor)
RMSE: 1.0474
MAE: 0.8027


Bước 5.4b: So sánh với baseline (bắt buộc để đánh giá công bằng)

In [9]:
global_mean = train_df['rating'].mean()
print("Global mean (train):", global_mean)

# Baseline 1: luôn đoán bằng điểm trung bình toàn cục
baseline_global_preds = [global_mean] * len(test_df)
rmse_baseline_global = np.sqrt(mean_squared_error(test_df['rating'], baseline_global_preds))
mae_baseline_global = mean_absolute_error(test_df['rating'], baseline_global_preds)
print(f"Baseline (global mean) - RMSE: {rmse_baseline_global:.4f}, MAE: {mae_baseline_global:.4f}")

# Baseline 2: đoán bằng điểm trung bình của từng user (mạnh hơn baseline 1)
user_mean = train_df.groupby('userId')['rating'].mean()
baseline_user_preds = test_df['userId'].map(user_mean).fillna(global_mean)
rmse_baseline_user = np.sqrt(mean_squared_error(test_df['rating'], baseline_user_preds))
mae_baseline_user = mean_absolute_error(test_df['rating'], baseline_user_preds)
print(f"Baseline (user mean) - RMSE: {rmse_baseline_user:.4f}, MAE: {mae_baseline_user:.4f}")

print()
print(f"So sánh: CF model RMSE={rmse:.4f} vs Baseline user-mean RMSE={rmse_baseline_user:.4f}")

Global mean (train): 3.579393738153544
Baseline (global mean) - RMSE: 1.0390, MAE: 0.8374
Baseline (user mean) - RMSE: 0.9543, MAE: 0.7443

So sánh: CF model RMSE=1.0474 vs Baseline user-mean RMSE=0.9543


Bước 5.4c: Sửa lại hàm dự đoán với mean-centering

In [11]:
# Tính sẵn trung bình rating của từng user (chỉ tính trên rating > 0 thực có trong train)
user_means_train = train_matrix.replace(0, np.nan).mean(axis=1)

def predict_rating_v2(user_id, tmdb_id, k_neighbors=10):
    if user_id not in train_matrix.index or tmdb_id not in train_matrix.columns:
        return None
    
    target_mean = user_means_train.get(user_id, global_mean)
    
    user_pos = train_matrix.index.get_loc(user_id)
    user_vector = train_matrix.iloc[user_pos].values.reshape(1, -1)
    
    distances, idx_knn = user_knn_train.kneighbors(user_vector, n_neighbors=k_neighbors + 1)
    similar_user_positions = idx_knn.flatten()[1:]
    similarities = 1 - distances.flatten()[1:]
    
    neighbor_ratings = train_matrix.iloc[similar_user_positions][tmdb_id].values
    neighbor_ids = train_matrix.index[similar_user_positions]
    neighbor_means = user_means_train.reindex(neighbor_ids).fillna(global_mean).values
    
    rated_mask = neighbor_ratings > 0
    if rated_mask.sum() == 0:
        return target_mean  # fallback: không có neighbor nào rate -> trả về trung bình của chính user
    
    weights = similarities[rated_mask]
    deviations = neighbor_ratings[rated_mask] - neighbor_means[rated_mask]
    
    if weights.sum() < 1e-9:
        return target_mean
    
    pred = target_mean + np.dot(weights, deviations) / weights.sum()
    return np.clip(pred, 0.5, 5.0)  # giữ trong khoảng thang rating hợp lệ

In [12]:
# Đánh giá lại với hàm mới (v2) — lưu ý: v2 luôn trả về giá trị (dùng target_mean làm fallback thay vì None)
predictions_v2, actuals_v2 = [], []

for _, row in test_df.iterrows():
    pred = predict_rating_v2(row['userId'], row['tmdbId'])
    predictions_v2.append(pred)
    actuals_v2.append(row['rating'])

rmse_v2 = np.sqrt(mean_squared_error(actuals_v2, predictions_v2))
mae_v2 = mean_absolute_error(actuals_v2, predictions_v2)
print(f"CF v2 (mean-centered) - RMSE: {rmse_v2:.4f}, MAE: {mae_v2:.4f}")
print(f"So sánh: CF v1={rmse:.4f} | CF v2={rmse_v2:.4f} | Baseline user-mean={rmse_baseline_user:.4f}")

CF v2 (mean-centered) - RMSE: 0.9818, MAE: 0.7455
So sánh: CF v1=1.0474 | CF v2=0.9818 | Baseline user-mean=0.9543


Bước 5.5: Precision@K, Recall@K cho cả 3 phương pháp

In [13]:
def get_relevant_items(user_id, threshold=4):
    """Phim mà user thực sự thích trong tập TEST (rating >= threshold)"""
    user_test = test_df[test_df['userId'] == user_id]
    return set(user_test[user_test['rating'] >= threshold]['tmdbId'])


def precision_recall_at_k(recommended_ids, relevant_ids, k=5):
    if len(relevant_ids) == 0:
        return None, None
    recommended_k = recommended_ids[:k]
    hits = len(set(recommended_k) & relevant_ids)
    precision = hits / k
    recall = hits / len(relevant_ids)
    return precision, recall

In [14]:
def get_cf_top_k_train(user_id, k=5, k_neighbors=10):
    """Lấy top-K gợi ý CF chỉ dựa trên train_matrix (không leak dữ liệu test)"""
    if user_id not in train_matrix.index:
        return []
    user_vector = train_matrix.loc[user_id].values.reshape(1, -1)
    distances, idx_knn = user_knn_train.kneighbors(user_vector, n_neighbors=k_neighbors + 1)
    sim_pos = idx_knn.flatten()[1:]
    sims = 1 - distances.flatten()[1:]
    weighted_scores = train_matrix.iloc[sim_pos].T.dot(sims) / (sims.sum() + 1e-9)
    already_rated = train_matrix.loc[user_id]
    candidates = weighted_scores[already_rated == 0]
    return candidates.sort_values(ascending=False).head(k).index.tolist()

In [15]:
# Đánh giá Precision@5, Recall@5 cho CF trên toàn bộ user có relevant items trong test
K = 5
precisions, recalls = [], []

test_users = test_df['userId'].unique()
for uid in test_users:
    relevant = get_relevant_items(uid, threshold=4)
    if len(relevant) == 0:
        continue
    recommended = get_cf_top_k_train(uid, k=K)
    p, r = precision_recall_at_k(recommended, relevant, k=K)
    if p is not None:
        precisions.append(p)
        recalls.append(r)

print(f"Số user đánh giá được: {len(precisions)} / {len(test_users)}")
print(f"Precision@{K}: {np.mean(precisions):.4f}")
print(f"Recall@{K}: {np.mean(recalls):.4f}")

Số user đánh giá được: 651 / 671
Precision@5: 0.2224
Recall@5: 0.1254


Bước 5.5b: Baseline Precision/Recall (gợi ý theo độ phổ biến, không cá nhân hoá)

In [16]:
# Baseline: gợi ý giống nhau cho mọi user — top phim theo weighted_rating (không cá nhân hoá)
top_popular_ids = movies.sort_values('weighted_rating', ascending=False)['id'].head(50).tolist()

def get_popularity_top_k(user_id, k=5):
    already_rated = set(train_df[train_df['userId'] == user_id]['tmdbId'])
    filtered = [mid for mid in top_popular_ids if mid not in already_rated]
    return filtered[:k]

precisions_pop, recalls_pop = [], []
for uid in test_users:
    relevant = get_relevant_items(uid, threshold=4)
    if len(relevant) == 0:
        continue
    recommended = get_popularity_top_k(uid, k=K)
    p, r = precision_recall_at_k(recommended, relevant, k=K)
    if p is not None:
        precisions_pop.append(p)
        recalls_pop.append(r)

print(f"Baseline (popularity) - Precision@{K}: {np.mean(precisions_pop):.4f}, Recall@{K}: {np.mean(recalls_pop):.4f}")
print(f"So sánh: CF Precision@{K}={np.mean(precisions):.4f} vs Baseline={np.mean(precisions_pop):.4f}")

Baseline (popularity) - Precision@5: 0.0482, Recall@5: 0.0227
So sánh: CF Precision@5=0.2224 vs Baseline=0.0482


Bước 5.5c: Precision/Recall cho Content-Based

In [17]:
movies_id_to_pos_local = pd.Series(movies.index, index=movies['id'])

def get_cb_top_k_train(user_id, k=5):
    """CB: dùng phim user đã thích trong TRAIN (rating>=4) làm seed, gợi ý theo candidate pool"""
    user_train = train_df[(train_df['userId'] == user_id) & (train_df['rating'] >= 4)]
    liked_ids = user_train['tmdbId'].tolist()
    if len(liked_ids) == 0:
        return []
    
    liked_pos = movies_id_to_pos_local.reindex(liked_ids).dropna().astype(int).values
    if len(liked_pos) == 0:
        return []
    
    sim_matrix = cosine_similarity(tfidf_matrix[liked_pos], tfidf_matrix[candidate_indices])
    cb_scores = sim_matrix.mean(axis=0)
    
    candidate_tmdb_ids_local = movies.iloc[candidate_indices]['id'].values
    already_rated = set(train_df[train_df['userId'] == user_id]['tmdbId'])
    
    scores_df = pd.DataFrame({'id': candidate_tmdb_ids_local, 'score': cb_scores})
    scores_df = scores_df[~scores_df['id'].isin(already_rated) & ~scores_df['id'].isin(liked_ids)]
    return scores_df.sort_values('score', ascending=False).head(k)['id'].tolist()


precisions_cb, recalls_cb = [], []
for uid in test_users:
    relevant = get_relevant_items(uid, threshold=4)
    if len(relevant) == 0:
        continue
    recommended = get_cb_top_k_train(uid, k=K)
    if len(recommended) == 0:
        continue
    p, r = precision_recall_at_k(recommended, relevant, k=K)
    if p is not None:
        precisions_cb.append(p)
        recalls_cb.append(r)

print(f"Số user đánh giá được (CB): {len(precisions_cb)}")
print(f"CB - Precision@{K}: {np.mean(precisions_cb):.4f}, Recall@{K}: {np.mean(recalls_cb):.4f}")

Số user đánh giá được (CB): 651
CB - Precision@5: 0.0065, Recall@5: 0.0046


Bước 5.5d: Sửa lại — giới hạn candidate pool của CB về cùng tập phim với CF để so sánh công bằng

In [18]:
# Giới hạn candidate pool CB chỉ trong phạm vi phim có mặt ở user_item_matrix (cùng "sân chơi" với CF)
cf_movie_ids = set(train_matrix.columns)
candidate_tmdb_ids_all = movies.iloc[candidate_indices]['id'].values
candidate_mask_fair = np.isin(candidate_tmdb_ids_all, list(cf_movie_ids))

candidate_indices_fair = candidate_indices[candidate_mask_fair]
candidate_tmdb_ids_fair = candidate_tmdb_ids_all[candidate_mask_fair]

print(f"Candidate pool công bằng (CB ∩ CF): {len(candidate_indices_fair)} phim")


def get_cb_top_k_fair(user_id, k=5):
    user_train = train_df[(train_df['userId'] == user_id) & (train_df['rating'] >= 4)]
    liked_ids = user_train['tmdbId'].tolist()
    if len(liked_ids) == 0:
        return []
    
    liked_pos = movies_id_to_pos_local.reindex(liked_ids).dropna().astype(int).values
    if len(liked_pos) == 0:
        return []
    
    sim_matrix = cosine_similarity(tfidf_matrix[liked_pos], tfidf_matrix[candidate_indices_fair])
    cb_scores = sim_matrix.mean(axis=0)
    
    already_rated = set(train_df[train_df['userId'] == user_id]['tmdbId'])
    
    scores_df = pd.DataFrame({'id': candidate_tmdb_ids_fair, 'score': cb_scores})
    scores_df = scores_df[~scores_df['id'].isin(already_rated) & ~scores_df['id'].isin(liked_ids)]
    return scores_df.sort_values('score', ascending=False).head(k)['id'].tolist()


precisions_cb2, recalls_cb2 = [], []
for uid in test_users:
    relevant = get_relevant_items(uid, threshold=4)
    if len(relevant) == 0:
        continue
    recommended = get_cb_top_k_fair(uid, k=K)
    if len(recommended) == 0:
        continue
    p, r = precision_recall_at_k(recommended, relevant, k=K)
    if p is not None:
        precisions_cb2.append(p)
        recalls_cb2.append(r)

print(f"Số user đánh giá được (CB công bằng): {len(precisions_cb2)}")
print(f"CB (fair) - Precision@{K}: {np.mean(precisions_cb2):.4f}, Recall@{K}: {np.mean(recalls_cb2):.4f}")
print()
print(f"So sánh cuối: CF={np.mean(precisions):.4f} | CB (fair)={np.mean(precisions_cb2):.4f} | Baseline popularity={np.mean(precisions_pop):.4f}")

Candidate pool công bằng (CB ∩ CF): 3459 phim
Số user đánh giá được (CB công bằng): 651
CB (fair) - Precision@5: 0.0280, Recall@5: 0.0146

So sánh cuối: CF=0.2224 | CB (fair)=0.0280 | Baseline popularity=0.0482


Bước 5.5e: Precision/Recall cho Hybrid

In [19]:
# --- 5.5e: Hybrid Precision/Recall ---
def get_hybrid_top_k_train(user_id, k=5, k_neighbors=10):
    cb_ids = get_cb_top_k_fair(user_id, k=len(candidate_indices_fair))  # toàn bộ candidate đã fair, xếp theo CB
    cf_scores_series = None
    if user_id in train_matrix.index:
        user_vector = train_matrix.loc[user_id].values.reshape(1, -1)
        distances, idx_knn = user_knn_train.kneighbors(user_vector, n_neighbors=k_neighbors + 1)
        sim_pos = idx_knn.flatten()[1:]
        sims = 1 - distances.flatten()[1:]
        cf_scores_series = train_matrix.iloc[sim_pos].T.dot(sims) / (sims.sum() + 1e-9)

    user_train = train_df[(train_df['userId'] == user_id) & (train_df['rating'] >= 4)]
    liked_ids = user_train['tmdbId'].tolist()
    liked_pos = movies_id_to_pos_local.reindex(liked_ids).dropna().astype(int).values

    if len(liked_pos) > 0:
        sim_matrix = cosine_similarity(tfidf_matrix[liked_pos], tfidf_matrix[candidate_indices_fair])
        cb_raw = sim_matrix.mean(axis=0)
    else:
        cb_raw = np.zeros(len(candidate_indices_fair))

    cf_raw = cf_scores_series.reindex(candidate_tmdb_ids_fair).fillna(0).values if cf_scores_series is not None else np.zeros(len(candidate_indices_fair))

    def norm(a):
        return np.zeros_like(a) if a.max() - a.min() < 1e-9 else (a - a.min()) / (a.max() - a.min())

    n_ratings = (train_matrix.loc[user_id] > 0).sum() if user_id in train_matrix.index else 0
    alpha = max(0.2, 1 - n_ratings / 50) if user_id in train_matrix.index else 1.0
    beta = 1 - alpha

    hybrid = alpha * norm(cb_raw) + beta * norm(cf_raw)
    already_rated = set(train_df[train_df['userId'] == user_id]['tmdbId'])

    scores_df = pd.DataFrame({'id': candidate_tmdb_ids_fair, 'score': hybrid})
    scores_df = scores_df[~scores_df['id'].isin(already_rated) & ~scores_df['id'].isin(liked_ids)]
    return scores_df.sort_values('score', ascending=False).head(k)['id'].tolist()


precisions_hy, recalls_hy = [], []
for uid in test_users:
    relevant = get_relevant_items(uid, threshold=4)
    if len(relevant) == 0:
        continue
    recommended = get_hybrid_top_k_train(uid, k=K)
    if len(recommended) == 0:
        continue
    p, r = precision_recall_at_k(recommended, relevant, k=K)
    if p is not None:
        precisions_hy.append(p)
        recalls_hy.append(r)

print(f"Hybrid - Precision@{K}: {np.mean(precisions_hy):.4f}, Recall@{K}: {np.mean(recalls_hy):.4f}")

Hybrid - Precision@5: 0.2341, Recall@5: 0.1261


Bước 5.6: Coverage/Diversity (chạy 1 lần)

In [21]:
# --- 5.6: Coverage & Diversity ---
def coverage_at_k(recommend_fn, k=5, sample_users=200):
    sampled = np.random.choice(test_users, size=min(sample_users, len(test_users)), replace=False)
    recommended_all = set()
    for uid in sampled:
        recs = recommend_fn(uid, k=k)
        recommended_all.update(recs)
    return len(recommended_all) / len(candidate_tmdb_ids_fair)

np.random.seed(42)
cov_cf = coverage_at_k(get_cf_top_k_train, k=K)
cov_cb = coverage_at_k(get_cb_top_k_fair, k=K)
cov_hy = coverage_at_k(get_hybrid_top_k_train, k=K)

print(f"Coverage@{K} (200 user mẫu) - CF: {cov_cf:.4f}, CB: {cov_cb:.4f}, Hybrid: {cov_hy:.4f}")

Coverage@5 (200 user mẫu) - CF: 0.0671, CB: 0.1223, Hybrid: 0.0752


Bước 5.7: So sánh CountVectorizer vs TF-IDF 

In [22]:
with open('../models_artifacts/count_matrix.pkl', 'rb') as f:
    count_matrix = pickle.load(f)

def get_cb_top_k_count(user_id, k=5):
    user_train = train_df[(train_df['userId'] == user_id) & (train_df['rating'] >= 4)]
    liked_ids = user_train['tmdbId'].tolist()
    if len(liked_ids) == 0:
        return []
    liked_pos = movies_id_to_pos_local.reindex(liked_ids).dropna().astype(int).values
    if len(liked_pos) == 0:
        return []
    sim_matrix = cosine_similarity(count_matrix[liked_pos], count_matrix[candidate_indices_fair])
    cb_scores = sim_matrix.mean(axis=0)
    already_rated = set(train_df[train_df['userId'] == user_id]['tmdbId'])
    scores_df = pd.DataFrame({'id': candidate_tmdb_ids_fair, 'score': cb_scores})
    scores_df = scores_df[~scores_df['id'].isin(already_rated) & ~scores_df['id'].isin(liked_ids)]
    return scores_df.sort_values('score', ascending=False).head(k)['id'].tolist()

precisions_count, recalls_count = [], []
for uid in test_users:
    relevant = get_relevant_items(uid, threshold=4)
    if len(relevant) == 0:
        continue
    recommended = get_cb_top_k_count(uid, k=K)
    if len(recommended) == 0:
        continue
    p, r = precision_recall_at_k(recommended, relevant, k=K)
    if p is not None:
        precisions_count.append(p)
        recalls_count.append(r)

print(f"CB (CountVectorizer) - Precision@{K}: {np.mean(precisions_count):.4f}, Recall@{K}: {np.mean(recalls_count):.4f}")
print(f"CB (TF-IDF)          - Precision@{K}: {np.mean(precisions_cb2):.4f}, Recall@{K}: {np.mean(recalls_cb2):.4f}")

CB (CountVectorizer) - Precision@5: 0.0108, Recall@5: 0.0089
CB (TF-IDF)          - Precision@5: 0.0280, Recall@5: 0.0146
